<a href="https://colab.research.google.com/github/SophiaPeritz/pinn-turbolence/blob/main/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Monta Drive per salvare i checkpoint
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/pinn-turbulence-results'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Cartella risultati: {DRIVE_DIR}")

# Clona la repo (la prima volta)
!git clone https://github.com/SophiaPeritz/pinn-turbolence.git
%cd pinn-turbolence

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'pinn-turbolence' already exists and is not an empty directory.
/content/pinn-turbolence


In [5]:
# Installa dipendenze
!pip install deepxde torch --quiet

In [6]:
# Importa i tuoi moduli
import sys
sys.path.append('/content/pinn-turbulence')

In [ ]:
# ── CELLA 4: Verifica GPU ─────────────────────────────────────
import torch
print('GPU disponibile:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('ATTENZIONE: nessuna GPU. Vai su Runtime > Cambia tipo di runtime > T4 GPU')

In [ ]:
# ── CELLA 5: Lancia il training ───────────────────────────────
import sys
sys.path.insert(0, '/content/pinn-turbolence')

from src.training import train, load_config

cfg = load_config('configs/kolmogorov.yaml')

# Salva i risultati su Google Drive invece che nella repo
cfg['results_dir'] = DRIVE_DIR + '/baseline'

model, history = train(cfg)

In [ ]:
# ── CELLA 6: Visualizza risultati ────────────────────────────
# (esegui questa cella dopo che il training e' finito)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(history['total'], color='black', label='Total')
axes[0].set_title('Loss totale')
axes[0].set_xlabel('Iterazione')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].semilogy(history['ic'],  color='blue',  label='IC loss')
axes[1].semilogy(history['pde'], color='red',   label='PDE loss')
axes[1].set_title('Componenti della loss')
axes[1].set_xlabel('Iterazione')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(DRIVE_DIR + '/baseline_loss.png', dpi=150)
plt.show()
print(f"Grafico salvato su Drive.")